# Fine-tune RoBERTa-base on M-DAIGT (EchoTrace single-instance AI-text detector)

Run this notebook in **Google Colab** (Runtime > Change runtime type > GPU). This machine has Hugging Face access, unlike the deployment laptop — see constraint 0.3 in the project brief. The final model is saved as a self-contained local folder that the deployment laptop loads with **no network access**.

**Data**: `train_mdaigt_task1.csv` (News Article Detection subtask, 10,000 real rows, 5,000 human / 5,000 machine). Provenance and verification against the RANLP 2025 M-DAIGT paper are documented in `docs/DATA_PROVENANCE.md` in the repo. This notebook reproduces the **exact same** stratified 80/10/10 split (seed=42) used by `scripts/build_sqlite.py`, so results here match what's in the SQLite data layer.

**Upload the data**: when prompted below, upload `data/raw/mdaigt/train_mdaigt_task1.csv` from your local repo checkout.

In [ ]:
!pip install -q -U "transformers>=4.46.0,<5.0.0" "accelerate>=0.26.0" datasets scikit-learn evaluate
!pip uninstall -y -q torchvision torchaudio
# Deliberately NOT upgrading torch: Colab ships a matched torch+torchvision
# pair, and upgrading only torch (as an earlier version of this cell did)
# leaves torchvision compiled against the old torch ABI, causing
# "RuntimeError: operator torchvision::nms does not exist" the moment
# transformers touches it. We don't need torchvision/torchaudio at all for
# text classification/QA, so removing them sidesteps the mismatch entirely
# rather than chasing matching version pairs.
#
# The -U on transformers/accelerate still matters: pip install without -U
# no-ops when a version is already present, silently leaving an older API
# in place (e.g. missing TrainingArguments' warmup_ratio). If you still hit
# a TypeError about an unexpected TrainingArguments keyword after running
# this cell, go to Runtime > Restart session and re-run from the top — a
# version already imported into this kernel won't be replaced just because
# pip updated the files on disk.

In [ ]:
from google.colab import files
print("Upload train_mdaigt_task1.csv")
uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

In [ ]:
import pandas as pd

SEED = 42
df = pd.read_csv(csv_path)
assert set(df["label"].unique()) == {"human", "machine"}
print(df.shape, df["label"].value_counts().to_dict())

# Identical stratified 80/10/10 split logic to scripts/build_sqlite.py — keep
# in sync so results here match the SQLite-recorded split.
train_parts, val_parts, test_parts = [], [], []
for label, group in df.groupby("label"):
    shuffled = group.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    n = len(shuffled)
    n_train = int(n * 0.8)
    n_val = int(n * 0.1)
    train_parts.append(shuffled.iloc[:n_train])
    val_parts.append(shuffled.iloc[n_train:n_train + n_val])
    test_parts.append(shuffled.iloc[n_train + n_val:])

train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_parts, ignore_index=True).reset_index(drop=True)
test_df = pd.concat(test_parts, ignore_index=True).reset_index(drop=True)

label2id = {"human": 0, "machine": 1}
for d in (train_df, val_df, test_df):
    d["labels"] = d["label"].map(label2id)

print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")

In [ ]:
from datasets import Dataset
from transformers import RobertaTokenizerFast

MODEL_NAME = "roberta-base"
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512, padding="max_length")

train_ds = Dataset.from_pandas(train_df[["text", "labels"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(val_df[["text", "labels"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df[["text", "labels"]]).map(tokenize, batched=True)

cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from scipy.special import softmax

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=-1)
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    auroc = roc_auc_score(labels, probs[:, 1])
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall, "auroc": auroc}

In [ ]:
from transformers import RobertaForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir="./roberta-mdaigt-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    num_train_epochs=20,  # capped upper bound; early stopping decides the real stopping point
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    fp16=True,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

## Training curves (auditable, not just a final number)

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_loss = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_loss = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]
eval_f1 = [(h["epoch"], h["eval_f1"]) for h in history if "eval_f1" in h]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(*zip(*train_loss), label="train_loss")
axes[0].plot(*zip(*eval_loss), label="eval_loss")
axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].set_title("Loss")
axes[1].plot(*zip(*eval_f1), label="eval_f1", color="green")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].set_title("Validation F1")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## Final held-out test evaluation (best checkpoint by validation, already loaded)

In [ ]:
test_metrics = trainer.evaluate(test_ds, metric_key_prefix="test")
print(test_metrics)

import json
with open("test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

## Save as a self-contained local folder (constraint 0.3)

This produces a folder that needs **no internet access** to reload — required so the deployment laptop can load it purely from a local path.

In [ ]:
SAVE_DIR = "./echotrace-detector"
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import shutil
shutil.copy("training_curves.png", SAVE_DIR)
shutil.copy("test_metrics.json", SAVE_DIR)

shutil.make_archive("echotrace-detector", "zip", SAVE_DIR)
files.download("echotrace-detector.zip")

## Next steps (back on your own machine)

1. Unzip `echotrace-detector.zip` into `models/echotrace-detector/` in the repo, replacing the empty placeholder folder.
2. Verify no-internet loading works:
   ```bash
   python3 -c "from transformers import RobertaForSequenceClassification, RobertaTokenizerFast; m=RobertaForSequenceClassification.from_pretrained('models/echotrace-detector'); t=RobertaTokenizerFast.from_pretrained('models/echotrace-detector'); print('loaded OK, no hub call')"
   ```
   (Optionally re-run with network disabled / `HF_HUB_OFFLINE=1` to prove it.)
3. `git add models/echotrace-detector` (already LFS-tracked via `*.safetensors` in `.gitattributes`) and commit.